In [ ]:
import os, json, subprocess, time, sys, importlib

print('=== 1. internet check ===')
try:
    r = subprocess.run(['curl','-sS','--max-time','5','https://huggingface.co/api/models/Qwen/Qwen3.6-35B-A3B'], capture_output=True, text=True, timeout=10)
    print('status:', r.returncode, 'bytes:', len(r.stdout))
    if r.stdout:
        meta = json.loads(r.stdout)
        print('  model id  :', meta.get('id'))
        print('  total params (BF16):', meta.get('safetensors',{}).get('total'))
        files = meta.get('siblings', [])
        st_files = [f for f in files if str(f.get('rfilename','')).endswith('.safetensors')]
        print('  safetensors shards:', len(st_files))
        for f in st_files[:5]:
            print('    -', f.get('rfilename'))
except Exception as e:
    print('internet check failed:', e)

print()
print('=== 2. system memory + disk ===')
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f'  RAM total : {vm.total/(1024**3):.1f} GB')
    print(f'  RAM avail : {vm.available/(1024**3):.1f} GB')
except Exception as e:
    print('psutil missing:', e)

for path in ['/kaggle/working','/kaggle/input','/tmp','/']:
    try:
        st = os.statvfs(path)
        free_gb = (st.f_bavail*st.f_frsize)/(1024**3)
        total_gb = (st.f_blocks*st.f_frsize)/(1024**3)
        print(f'  disk {path:<16} {free_gb:>7.1f} GB free / {total_gb:>7.1f} GB total')
    except Exception as e:
        print(f'  disk {path}: {e}')

print()
print('=== 3. test kagglehub HF bridge ===')
try:
    import kagglehub
    print('kagglehub version:', getattr(kagglehub,'__version__','?'))
    print('functions:', [x for x in dir(kagglehub) if not x.startswith('_')])
except Exception as e:
    print('kagglehub import failed:', e)

print()
print('=== 4. test huggingface_hub snapshot of small file ===')
try:
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi()
    print('hf_api type:', type(api).__name__)
    cfg_path = hf_hub_download('Qwen/Qwen3.6-35B-A3B', 'config.json', cache_dir='/tmp/hf_cache')
    print('downloaded config to:', cfg_path)
    cfg = json.load(open(cfg_path))
    print('  arch       :', cfg.get('architectures'))
    print('  hidden_size:', cfg.get('hidden_size'))
    print('  num layers :', cfg.get('num_hidden_layers'))
    print('  vocab_size :', cfg.get('vocab_size'))
    print('  num_experts:', cfg.get('num_experts'))
    print('  experts/tok:', cfg.get('num_experts_per_tok'))
    print('  max_pos    :', cfg.get('max_position_embeddings'))
except Exception as e:
    print('hf_hub_download failed:', repr(e))

print()
print('=== 5. list available Kaggle Models for offline mounting ===')
# Try to attach the qwen-3-5 model from the model_sources in metadata at kernel push time
# We do not attach a model_sources here, just verify what would be available offline.
print('check /kaggle/input/ tree:')
for r in subprocess.run(['ls','-la','/kaggle/input/'], capture_output=True, text=True).stdout.splitlines():
    print('  ', r)

print()
print('=== 6. Kaggle Dataset size policy probe ===')
# We just print what Kaggle docs say at this point in time (per public docs, individual Datasets <= 500GB; private <= 100GB)
print('Per Kaggle docs (2025): public Dataset cap is 500GB, private cap is 100GB.')
print('Per Kaggle docs (2025): Models are size-uncapped but each instance has its own size.')
